# (연구) HST example 3
> 완성

- toc:true
- branch: master
- badges: true
- comments: true
- author: 신록예찬
- hide: false
- categories: [기하학적 딥러닝, 논문연구]

### import

In [26]:
## 1. remove trash
!rm -rf ~/.local/share/Trash/files/*
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rc('image', cmap='Greys')
import rpy2 
%load_ext rpy2.ipython
%run pybase
%run heavysnow 

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


### load data

In [27]:
n=23
f=np.array(pd.read_csv("2021-08-15-MCU-ticket.csv").Worldwide)/1000000
V=np.array(pd.read_csv("2021-08-15-MCU-ticket.csv").Film)
W=np.array(pd.read_csv("2021-08-15-MCU-weights.csv",index_col=0))-np.eye(n,n)

### HST

In [28]:
gs=GraphSignal(V,W,f)
hst=HeavySnowTransform(gs)
hst.snow(tau=200000)

HST (tau= 200000, b=1)
200000/200000
HST completed and all history is recorded.


In [ ]:
#hstrslt=pd.read_csv('hstrslt.csv')
Σ=sigmamat(hh)
Wτ=weightmat(Σ,θ=np.mean(Σ)/28)
#print(np.sum(Wτ))

In [ ]:
sdf

### 유클리드, 그래프도메인만 반영한 웨이트 정리

In [ ]:
## 유클리드
WEuclid=np.exp(-sigmamat(f)/np.mean(f))-np.diag([1]*n)
## 그래프 
Wgraph=(W+W.T)/2
## hst 
Whst=Wτ

from matplotlib.pyplot import imshow as image
image(WEuclid)
image(Wgraph)
image(Whst)

### R환경으로 push

In [ ]:
#push(hh,"hh")
push(n,"n")
push(V,"V")
push(f,"f")
push(WEuclid,"WEuclid")
push(Wgraph,"Wgraph")
push(Whst,"Whst")
push(W,"W")

### 유클리드와 그래프도메인의 정보를 플랏. 

In [ ]:
%%R -r 150 -w 2000 -h 660
dat_<-tibble(V)
dat_$f<-as.vector(f)
dat_$divlink<-0
dat_$conlink<-0
dat_$link<-0
dat<-dat_
for(i in 1:23){
    dat$divlink[i]<-sum(W[i,])
    dat$conlink[i]<-sum(W[,i])
    dat$link[i]<-sum(W[i,])+sum(W[i,])
}
p1_<-ggplot(dat)+geom_point(aes(x=f,y=divlink),size=3)+theme_light()+
        geom_text_repel(aes(x=f,y=divlink,label=V),col="gray40")+
        ylab("Out-degree")+xlab("Box office record")+
        geom_hline(aes(yintercept=2),col=2,lty=2,lwd=0.5)+
        geom_vline(aes(xintercept=1000),col=2,lty=2,lwd=0.5)+
        ggtitle("(a)")+theme(plot.title=element_text(face="bold.italic",size=rel(2)))+
        theme(axis.title.x=element_text(face=4,size=rel(1)))+
        theme(axis.title.y=element_text(face=4,size=rel(1)))
p2_<-ggplot(dat)+geom_point(aes(x=f,y=conlink),size=3)+theme_light()+
        geom_text_repel(aes(x=f,y=conlink,label=V),col="gray40")+
        ylab("In-degree")+xlab("Box office record")+
        geom_hline(aes(yintercept=3.75),col=2,lty=2,lwd=0.5)+
        geom_vline(aes(xintercept=1000),col=2,lty=2,lwd=0.5)+
        ggtitle("(b)")+theme(plot.title=element_text(face="bold.italic",size=rel(2)))+
        theme(axis.title.x=element_text(face=4,size=rel(1)))+
        theme(axis.title.y=element_text(face=4,size=rel(1)))
# p3_<-ggplot(dat)+geom_point(aes(x=f,y=link),size=3)+theme_light()+
#         geom_text_repel(aes(x=f,y=link,label=V),col="gray40")+
#         ylab("Graph")+xlab("Box office record")+
#         geom_hline(aes(yintercept=6.25),col=2,lty=2,lwd=0.5)+
#         geom_vline(aes(xintercept=1000),col=2,lty=2,lwd=0.5)+
#         ggtitle("(c)")
p1<-grid.arrange(p1_,p2_,ncol=2)
ggsave(plot=p1,"./fig/p1.pdf",width=9,height=4.5)

### gfft 수행하고 결과를 저장함. 

In [ ]:
%%R 
g1<-gfft(f,WEuclid)
g2<-gfft(f,Wgraph)
g3<-gfft(f,Whst)

### 아이겐플랏 


In [ ]:
%%R -r 150 -w 1000 -h 400
library(gridExtra)
e1<-eigenplot(g1)+ggtitle("(a) Euclid weights")+theme(plot.title=element_text(face="bold.italic",size=rel(1)))
e2<-eigenplot(g2)+ggtitle("(b) Graph weights")+theme(plot.title=element_text(face="bold.italic",size=rel(1)))
e3<-eigenplot(g3)+ggtitle("(c) HST weights")+theme(plot.title=element_text(face="bold.italic",size=rel(1)))
s1<-specplot(g1)
s2<-specplot(g2)
s3<-specplot(g3)
p2<-grid.arrange(e1,e2,e3,s1,s2,s3,nrow=2)
ggsave(plot=p2,"./fig/p2.pdf",width=6,height=3)

### 분해 

In [ ]:
%%R -r 300 -w 2500 -h 1500
#source("ex1.r")
d1<-decompose(f,WEuclid,V=V,showingeigenvector=F) # 0, 35000, 60000, 80000
d2<-decompose(f,Wgraph,V=V,showingeigenvector=F) # 0, 35000, 60000, 80000
d3<-decompose(f,Whst,V=V,showingeigenvector=F) # 0, 35000, 60000, 80000

d1$case<-"Euclid"
d2$case<-"Graph"
d3$case<-"HST"

decomprslt<-rbind(d1,d2,d3)

In [ ]:
%%R -r 300 -w 1800 -h 2200
decomp_dat<- decomprslt %>% 
                group_by(case,eigenvectorindex) %>% 
                mutate(textsize=(fhat>mean(fhat)) *(fhat>320) * ((case=="Euclid")*1.7+(case!="Euclid")*(eigenvectorindex!=1)*1.7+(case!="Euclid")*(eigenvectorindex==1)*1))
p3<-ggplot(data=filter(decomp_dat,eigenvectorindex %in% 1:7))+
    geom_col(aes(x=Vindex,y=fhat,fill=fhat>0),width=0.7)+
    geom_text_repel(aes(x=Vindex,y=fhat,label=V,size=textsize),col=1,fontface=4,alpha=0.8,segment.size=0.2,segment.color="gray60",min.segment.length=5,hjust=0.1)+
    scale_radius(range = c(0,1.8))+
    guides(size=FALSE)+
    facet_grid(eigenvectorindex~case)+
    geom_hline(aes(yintercept=0),col="gray60",lty=2)+
    xlab("")+ylab("")+guides(fill=FALSE)+
    theme(axis.text.x=element_text(angle=85,hjust=1,vjust=1,face=4,size=rel(0.7),colour="gray60"))+
    theme_light()+
    theme(strip.text.x = element_text(size = 10, color = "black", face = "bold.italic"))+
    theme(strip.text.y = element_text(size = 10, color = "black", face = "bold.italic"))+
    theme(plot.title=element_text(face="bold.italic",size=rel(1.5)))
show(p3)
ggsave(plot=p3,"./fig/p3.pdf",width=6,height=6)

In [ ]:
%%R
dat3_Euclid <- decomp_dat %>% filter(case=="Euclid",eigenvectorindex==1) %>% select(V,case,eigenvectorindex,textsize,fhat)
dat3_Euclid <- left_join(dat,dat3_Euclid,by="V")

for(j in 2:23){
        decomp_filtered_dat_<- decomp_dat %>% filter(case=="Euclid",eigenvectorindex==j) %>% select(V,case,eigenvectorindex,textsize,fhat)
        decomp_filtered_dat_<- left_join(dat,decomp_filtered_dat_,by="V")
        dat3_Euclid<-rbind(dat3_Euclid,decomp_filtered_dat_)
}

dat3_Graph <- decomp_dat %>% filter(case=="Graph",eigenvectorindex==1) %>% select(V,case,eigenvectorindex,textsize,fhat)
dat3_Graph <- left_join(dat,dat3_Graph,by="V")

for(j in 2:23){
        decomp_filtered_dat_<- decomp_dat %>% filter(case=="Graph",eigenvectorindex==j) %>% select(V,case,eigenvectorindex,textsize,fhat)
        decomp_filtered_dat_<- left_join(dat,decomp_filtered_dat_,by="V")
        dat3_Graph<-rbind(dat3_Graph,decomp_filtered_dat_)
}

dat3_HST <- decomp_dat %>% filter(case=="HST",eigenvectorindex==1) %>% select(V,case,eigenvectorindex,textsize,fhat)
dat3_HST <- left_join(dat,dat3_HST,by="V")

for(j in 2:23){
        decomp_filtered_dat_<- decomp_dat %>% filter(case=="HST",eigenvectorindex==j) %>% select(V,case,eigenvectorindex,textsize,fhat)
        decomp_filtered_dat_<- left_join(dat,decomp_filtered_dat_,by="V")
        dat3_HST<-rbind(dat3_HST,decomp_filtered_dat_)
}
dat3<-rbind(dat3_Euclid,dat3_Graph,dat3_HST)

In [ ]:
%%R -r 300 -w 2000 -h 3000
hull_cyl <- dat3 %>%
            group_by(case,eigenvectorindex) %>% 
            filter(textsize>0) %>% 
            slice(chull(f, divlink))
p4<-ggplot(dat3%>% filter(eigenvectorindex<8))+geom_point(aes(x=f,y=divlink,col=textsize>0,size=fhat*(fhat>0)),alpha=0.8)+
        geom_shape(data=hull_cyl%>% filter(eigenvectorindex<8) ,aes(x=f,y=divlink),alpha=0.05,col="gray30",radius=0.05,expand=0.05)+
        facet_grid(eigenvectorindex~case)+
        scale_size_continuous(range=c(2,2.5))+
        scale_color_manual(values=c("gray80", "gray20"))+
        ylim(0,8)+xlim(0,3000)+
        guides(alpha=F)+
        guides(col=F)+
        guides(size=F)+
        ylab("")+xlab("")+
        geom_hline(aes(yintercept=2),col="gray30",lty=2,lwd=0.2)+
        geom_vline(aes(xintercept=1000),col="gray30",lty=2,lwd=0.2)+
        theme_light()+
        theme(strip.text.x = element_text(size = 10, color = "black", face = "bold.italic"))+
        theme(strip.text.y = element_text(size = 10, color = "black", face = "bold.italic"))+
        theme(plot.title=element_text(face="bold.italic",size=rel(1.5)))
show(p4)
ggsave(plot=p4,"./fig/p4.pdf",width=7,height=7)

In [ ]:
%%R -r 300 -w 2000 -h 3000
hull_cyl <- dat3 %>%
            group_by(case,eigenvectorindex) %>% 
            filter(textsize>0) %>% 
            slice(chull(f, conlink))
p5<-ggplot(dat3%>% filter(eigenvectorindex<8))+geom_point(aes(x=f,y=conlink,col=textsize>0,size=fhat*(fhat>0)),alpha=0.8)+
        geom_shape(data=hull_cyl%>% filter(eigenvectorindex<8) ,aes(x=f,y=conlink),alpha=0.05,col="gray30",radius=0.05,expand=0.05)+
        facet_grid(eigenvectorindex~case)+
        scale_size_continuous(range=c(2,2.5))+
        scale_color_manual(values=c("gray80", "gray20"))+        
        ylim(0,10)+xlim(0,3000)+
        guides(col=F)+
        guides(alpha=F)+
        guides(size=F)+       
        ylab("")+xlab("")+
        geom_hline(aes(yintercept=3.75),col="gray30",lty=2,lwd=0.2)+
        geom_vline(aes(xintercept=1000),col="gray30",lty=2,lwd=0.2)+
        theme_light()+
        theme(strip.text.x = element_text(size = 10, color = "black", face = "bold.italic"))+
        theme(strip.text.y = element_text(size = 10, color = "black", face = "bold.italic"))+
        theme(plot.title=element_text(face="bold.italic",size=rel(1.5)))
show(p5)
ggsave(plot=p5,"./fig/p5.pdf",width=7,height=7)        

In [ ]:
%%R 
p_<-grid.arrange(p4,p5,ncol=2)
p6<-grid.arrange(p1,p_,nrow=2,heights=c(3,5))
ggsave(plot=p6,"./fig/p6.pdf",width=10,height=12)